# 🏰 RAG con soberanía total: Postgres + pgvector + Gemma, todo tuyo

**Curso práctico · ~90 minutos · Google Colab (GPU) + una VM de GCloud**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_selfhosted_pgvector.ipynb)

Último paso de la serie. Hasta ahora, cada curso quitó una dependencia: el curso 3 sustituyó el OCR de Google por un modelo abierto en tu GPU. **Ahora quitamos las tres que quedaban** — el almacén de vectores, los embeddings y el modelo que genera — y las montamos **en infraestructura propia**.

| Pieza | Antes (servicio de Google) | **Ahora (tuyo)** |
|---|---|---|
| Almacén de vectores | BigQuery | **PostgreSQL + `pgvector`** en una VM de Compute Engine |
| Embeddings | Vertex `ML.GENERATE_EMBEDDING` | **`BAAI/bge-m3`** (open-source) en la GPU |
| Generación | Gemini | **Gemma 4 (E4B)** en la GPU, en 4-bit |

> ⚠️ **Necesitas GPU** (*Entorno de ejecución → GPU*; T4 vale) **y un proyecto GCP con facturación** (crearemos una VM pequeña, `e2-medium`, y **la borraremos al final**).

> El único "tercero" que queda es **GCloud como IaaS** (nos alquila la VM). Ningún servicio de IA gestionado: el dato y los modelos son tuyos, y el PDF nunca sale de tu perímetro.


## 🎒 Kit de supervivencia (si es tu primera vez con RAG)

- **Embedding** — texto → **vector** de números que captura su significado; textos parecidos, vectores cercanos.
- **Vector store** — una base de datos que guarda esos vectores y sabe encontrar los más cercanos a una consulta. Antes era BigQuery; **hoy es Postgres con la extensión `pgvector`**.
- **Chunk** — trozo de documento que se vectoriza por separado.
- **RAG** — **(1)** recuperar los chunks relevantes, **(2)** pegárselos al modelo como contexto, **(3)** generar la respuesta con citas.

> El detalle a fondo está en los cursos anteriores ([emails](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_emails_bigquery_v2.ipynb), [PDF con Document AI](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_pdf_polizas_bigquery.ipynb), [PDF con OCR open-source](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_pdf_ocr_opensource.ipynb)).


---
## ⚖️ ¿Por qué montarlo todo uno mismo?

Un servicio gestionado (BigQuery, Vertex, Gemini) es cómodo: cero infraestructura, escala solo. Pero tiene un precio que en algunos dominios es inaceptable:

- **El dato sale de tu perímetro.** En seguros, salud o banca, mandar documentos con datos personales a una API de terceros choca de frente con el RGPD y con muchos contratos.
- **Dependes de un proveedor**: su precio, su disponibilidad, sus cambios de API, su decisión de discontinuar un modelo.
- **Coste por uso** que, a volumen, se dispara.

Montarlo uno mismo cuesta trabajo e infraestructura, pero te da **soberanía**: el dato no se mueve, el modelo es tuyo (pesos que puedes guardar), y el coste es el de una máquina que controlas.

> 🎓 **La tesis de este curso:** no es que self-hosted sea "mejor" — es que **es una decisión de arquitectura con trade-offs**, y saber montarlo te da la opción. Al final (bloque 7) los comparamos con honestidad.

### Requisitos y agenda

- GPU en Colab + proyecto GCP con facturación.
- No hace falta BigQuery, ni Vertex, ni Document AI.

| # | Bloque | ⏱️ |
|---|--------|----|
| 0 | Setup | 8 min |
| 1 | Fabricar las pólizas en PDF | 5 min |
| 2 | Extraer texto y trocear | 8 min |
| 3 | **Levantar Postgres + pgvector en una VM** | 20 min |
| 4 | **Embeddings open-source → pgvector** | 12 min |
| 5 | **Búsqueda vectorial en SQL** (`<=>`) | 8 min |
| 6 | **RAG con Gemma 4 en tu GPU** | 15 min |
| 7 | Comparativa self-hosted vs. gestionado | 6 min |
| 8 | Evaluación y **borrar la VM** | 8 min |


---
# 0 · Setup ⏱️ ~8 min

▶️ **Qué hace esta celda:** instala todo lo local. Nada de clientes de BigQuery ni Vertex: ahora tenemos `sentence-transformers` (embeddings), `psycopg2` (hablar con Postgres), `bitsandbytes` (cargar Gemma en 4-bit) y `transformers` **actualizado** (Gemma 4 necesita una versión reciente). No tocamos `torch`.

In [ ]:
%pip install -q -U transformers accelerate bitsandbytes sentence-transformers \
    psycopg2-binary reportlab pypdf
print("✅ Dependencias instaladas")

▶️ **Qué hace esta celda:** comprueba la GPU.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ No hay GPU. Entorno de ejecución → Cambiar tipo de entorno → GPU.")
gpu = torch.cuda.get_device_properties(0)
print(f"✅ GPU: {gpu.name} · {gpu.total_memory/1e9:.0f} GB")

▶️ **Qué hace esta celda:** te autentica en GCloud y habilita las APIs que usaremos para la VM: **Compute Engine** (la máquina) e **IAP** (el túnel seguro para conectarnos a Postgres sin abrirlo a internet).

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("✅ Autenticado")

# 👇 EDITA con tu proyecto
PROJECT_ID = "tu-proyecto-gcp"  # @param {type:"string"}
!gcloud config set project {PROJECT_ID} --quiet
!gcloud services enable compute.googleapis.com iap.googleapis.com --quiet
print("✅ APIs Compute + IAP habilitadas")

> 🚩 **CHECKPOINT 1** — GPU detectada, proyecto fijado, APIs habilitadas.

---
# 1 · Fabricamos las pólizas en PDF ⏱️ ~5 min

Idéntico a los cursos de PDF: tres condicionados con su numeración, tablas y —clave— la sección **4. EXCLUSIONES** con texto parecido al de **3. COBERTURAS**.

▶️ **Qué hace esta celda:** estilos y plantilla del documento.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (BaseDocTemplate, PageTemplate, Frame, Paragraph,
                                Spacer, Table, TableStyle, PageBreak)
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER

ASEGURADORA = "Peñalara Seguros, S.A."

ss = getSampleStyleSheet()
H1 = ParagraphStyle("H1x", parent=ss["Heading1"], fontSize=15, spaceAfter=10,
                    textColor=colors.HexColor("#1a3d5c"))
H2 = ParagraphStyle("H2x", parent=ss["Heading2"], fontSize=12, spaceBefore=10,
                    spaceAfter=6, textColor=colors.HexColor("#2c5f8a"))
H3 = ParagraphStyle("H3x", parent=ss["Heading3"], fontSize=10.5, spaceBefore=8,
                    spaceAfter=4, textColor=colors.HexColor("#444444"))
BODY = ParagraphStyle("BODYx", parent=ss["BodyText"], fontSize=9.5, leading=13,
                      alignment=TA_JUSTIFY, spaceAfter=5)
# 👇 la famosa "letra pequeña": legalmente válida, visualmente hostil
SMALL = ParagraphStyle("SMALLx", parent=BODY, fontSize=6.5, leading=8.5,
                       textColor=colors.HexColor("#555555"))
TITLE = ParagraphStyle("TITLEx", parent=ss["Title"], fontSize=22,
                       textColor=colors.HexColor("#1a3d5c"))
CENTER = ParagraphStyle("CENTERx", parent=BODY, alignment=TA_CENTER)

def _tabla(data, col_widths):
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1a3d5c")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#eef3f8")]),
    ]))
    return t

class PolizaDoc(BaseDocTemplate):
    """Documento con encabezado y pie 'Página X de Y' en cada página."""
    def __init__(self, filename, producto, codigo, **kw):
        super().__init__(filename, pagesize=A4, **kw)
        self.producto, self.codigo = producto, codigo
        frame = Frame(2.2*cm, 2.2*cm, A4[0]-4.4*cm, A4[1]-4.4*cm, id="n")
        self.addPageTemplates([PageTemplate(id="all", frames=[frame], onPage=self._decorar)])

    def _decorar(self, canvas, doc):
        canvas.saveState()
        canvas.setFont("Helvetica", 7)
        canvas.setFillColor(colors.HexColor("#777777"))
        canvas.drawString(2.2*cm, A4[1]-1.5*cm, f"{ASEGURADORA} · {self.producto}")
        canvas.drawRightString(A4[0]-2.2*cm, A4[1]-1.5*cm, f"Condicionado {self.codigo}")
        canvas.line(2.2*cm, A4[1]-1.65*cm, A4[0]-2.2*cm, A4[1]-1.65*cm)
        canvas.line(2.2*cm, 1.9*cm, A4[0]-2.2*cm, 1.9*cm)
        canvas.drawCentredString(A4[0]/2.0, 1.4*cm, f"Página {doc.page}")
        canvas.restoreState()

print("✅ Estilos y plantilla definidos")

▶️ **Qué hace esta celda:** la función que arma cada póliza.

In [ ]:
def construir_poliza(path, producto, codigo, version, coberturas, exclusiones, franquicias):
    story = []
    # ── Portada ──
    story += [Spacer(1, 4*cm), Paragraph(ASEGURADORA, CENTER), Spacer(1, 1*cm),
              Paragraph(producto, TITLE), Spacer(1, 0.6*cm),
              Paragraph("Condiciones Generales", CENTER), Spacer(1, 0.3*cm),
              Paragraph(f"Código de condicionado: <b>{codigo}</b> · Versión {version}", CENTER),
              PageBreak()]

    # ── 1. Definiciones ──
    story += [Paragraph("1. DEFINICIONES", H1),
              Paragraph("A efectos del presente contrato, se entiende por:", BODY)]
    for num, term, txt in [
        ("1.1", "Asegurado", "Persona física o jurídica titular del interés asegurado y sobre la que recaen las consecuencias económicas del siniestro."),
        ("1.2", "Tomador", "Persona que suscribe el contrato con el Asegurador y a quien corresponden las obligaciones derivadas del mismo."),
        ("1.3", "Siniestro", "Todo hecho cuyas consecuencias estén total o parcialmente cubiertas por las garantías de esta póliza."),
        ("1.4", "Franquicia", "Cantidad que queda a cargo del Asegurado en cada siniestro y que se deduce de la indemnización."),
        ("1.5", "Suma asegurada", "Límite máximo de indemnización por siniestro y anualidad de seguro."),
    ]:
        story += [Paragraph(f"{num} {term}", H3), Paragraph(txt, BODY)]

    # ── 2. Objeto ──
    story += [PageBreak(), Paragraph("2. OBJETO DEL SEGURO", H1),
              Paragraph(f"El Asegurador garantiza, dentro de los límites del presente condicionado, "
                        f"las consecuencias económicas de los riesgos descritos en la sección 3, hasta "
                        f"las sumas fijadas en las Condiciones Particulares de la póliza {producto}.", BODY),
              Paragraph("2.1 Ámbito territorial", H3),
              Paragraph("Las garantías surten efecto en el territorio español, salvo indicación expresa "
                        "en contrario en las Condiciones Particulares.", BODY),
              Paragraph("2.2 Ámbito temporal", H3),
              Paragraph("Quedan cubiertos los siniestros ocurridos durante la vigencia de la póliza y "
                        "declarados conforme a los plazos de la sección 6.", BODY)]

    # ── 3. Coberturas (CON TABLA) ──
    story += [PageBreak(), Paragraph("3. COBERTURAS", H1),
              Paragraph("Quedan cubiertas las siguientes garantías, con los límites indicados:", BODY),
              Spacer(1, 0.3*cm),
              _tabla([["Garantía", "Límite por siniestro", "Franquicia"]] + coberturas,
                     [7.5*cm, 4.5*cm, 3.5*cm]), Spacer(1, 0.4*cm)]
    for i, (gar, lim, fr) in enumerate(coberturas, start=1):
        story += [Paragraph(f"3.{i} {gar}", H3),
                  Paragraph(f"Se garantiza el pago de la indemnización por los daños directos "
                            f"ocasionados por {gar.lower()}, hasta el límite de {lim} por siniestro, "
                            f"con una franquicia de {fr}. La cobertura opera siempre que el hecho "
                            f"causante sea súbito, accidental e imprevisto para el Asegurado.", BODY)]

    # ── 4. Exclusiones (🎯 el gotcha) ──
    story += [PageBreak(), Paragraph("4. EXCLUSIONES", H1),
              Paragraph("<b>Con carácter general, y salvo pacto expreso en contrario, quedan "
                        "EXCLUIDOS de toda cobertura:</b>", BODY)]
    for i, (titulo, items) in enumerate(exclusiones, start=1):
        story += [Paragraph(f"4.{i} {titulo}", H2)]
        for letra, txt in zip("abcdefghij", items):
            story += [Paragraph(f"4.{i}.{letra}) {txt}", BODY)]
    story += [Spacer(1, 0.3*cm),
              Paragraph("Las exclusiones recogidas en la presente sección han sido específicamente "
                        "aceptadas por el Tomador mediante su firma en las Condiciones Particulares, "
                        "conforme al artículo 3 de la Ley 50/1980, de Contrato de Seguro, que exige "
                        "que las cláusulas limitativas de los derechos del Asegurado se destaquen de "
                        "modo especial y sean expresamente aceptadas por escrito.", SMALL)]

    # ── 5. Franquicias (otra tabla) ──
    story += [PageBreak(), Paragraph("5. FRANQUICIAS", H1),
              Paragraph("Se aplicarán las siguientes franquicias por modalidad:", BODY),
              Spacer(1, 0.3*cm),
              _tabla([["Modalidad", "Franquicia general", "Franquicia específica"]] + franquicias,
                     [6*cm, 4.75*cm, 4.75*cm])]

    # ── 6. Siniestros ──
    story += [PageBreak(), Paragraph("6. DECLARACIÓN Y TRAMITACIÓN DE SINIESTROS", H1),
              Paragraph("6.1 Plazo de comunicación", H3),
              Paragraph("El Tomador deberá comunicar el siniestro al Asegurador en el plazo máximo de "
                        "<b>siete (7) días</b> desde que tuviera conocimiento del mismo.", BODY),
              Paragraph("6.2 Documentación exigible", H3),
              Paragraph("Deberá aportarse: declaración del siniestro, acreditación de la titularidad "
                        "del bien, presupuesto o factura de reparación y, cuando proceda, atestado.", BODY),
              Paragraph("6.3 Peritación", H3),
              Paragraph("En caso de desacuerdo sobre la valoración, cada parte designará un perito. De "
                        "persistir la discrepancia, se designará un tercer perito de común acuerdo.", BODY),
              Paragraph("6.4 Pago de la indemnización", H3),
              Paragraph("El Asegurador abonará la indemnización en el plazo de cuarenta (40) días desde "
                        "la recepción de la declaración del siniestro.", BODY)]

    # ── 7. Prima ──
    story += [PageBreak(), Paragraph("7. PRIMA, DURACIÓN Y RENOVACIÓN", H1),
              Paragraph("7.1 Pago de la prima", H3),
              Paragraph("La prima es anual y pagadera por anticipado.", BODY),
              Paragraph("7.2 Impago y suspensión", H3),
              Paragraph("En caso de impago de la segunda o sucesivas primas, la cobertura quedará "
                        "<b>suspendida un mes después</b> del día de su vencimiento.", BODY),
              Paragraph("7.3 Duración y prórroga", H3),
              Paragraph("El contrato se prorrogará tácitamente por periodos anuales, salvo oposición "
                        "notificada con <b>un (1) mes</b> de antelación por el Tomador o <b>dos (2) "
                        "meses</b> por el Asegurador.", BODY)]

    PolizaDoc(path, producto, codigo).multiBuild(story)
    return path

print("✅ Constructor de pólizas definido")

▶️ **Qué hace esta celda:** el catálogo de 3 productos y la generación.

In [ ]:
import os

POLIZAS = [
    dict(
        path="HOGAR_PLUS.pdf", producto="Hogar Plus", codigo="HP-2026-01", version="3.2",
        coberturas=[
            ["Incendio, rayo y explosión", "300.000 €", "Sin franquicia"],
            ["Daños por agua por rotura accidental de conducciones", "50.000 €", "150 €"],
            ["Robo y expoliación en el interior de la vivienda", "30.000 €", "150 €"],
            ["Rotura de cristales y vitrocerámica", "3.000 €", "Sin franquicia"],
            ["Responsabilidad civil familiar", "150.000 €", "300 €"],
            ["Fenómenos atmosféricos (viento, pedrisco, nieve)", "100.000 €", "300 €"],
        ],
        exclusiones=[
            ("Daños por agua no cubiertos", [
                "Los daños causados por <b>humedades, condensación o filtraciones</b> a través de muros, "
                "fachadas, terrazas o cubiertas, aun cuando sean consecuencia de lluvia, nieve o granizo.",
                "Los daños derivados de <b>falta de mantenimiento</b> de las conducciones, así como la "
                "corrosión, el óxido o el desgaste paulatino de tuberías.",
                "El coste de <b>localización y reparación de la avería</b> cuando no se haya producido "
                "daño material indemnizable.",
                "Los daños por <b>agua de lluvia que penetre por ventanas, puertas o huecos dejados "
                "abiertos</b> o defectuosamente cerrados por el Asegurado.",
            ]),
            ("Exclusiones generales", [
                "Los daños causados con dolo o culpa grave del Asegurado.",
                "Los daños derivados de <b>vicio propio o defecto de construcción</b> preexistente.",
                "Los daños calificados como catástrofe nacional o cubiertos por el <b>Consorcio de "
                "Compensación de Seguros</b>.",
                "Los daños en <b>viviendas deshabitadas</b> más de 60 días consecutivos.",
            ]),
        ],
        franquicias=[["Vivienda habitual", "150 €", "300 € en RC familiar"],
                     ["Segunda residencia", "300 €", "600 € en daños por agua"],
                     ["Vivienda en alquiler", "300 €", "600 € en robo"]],
    ),
    dict(
        path="AUTO_TODO_RIESGO.pdf", producto="Auto Todo Riesgo", codigo="AT-2026-04", version="2.1",
        coberturas=[
            ["Responsabilidad civil obligatoria", "Ilimitada (legal)", "Sin franquicia"],
            ["Daños propios por colisión o vuelco", "Valor venal + 20%", "300 €"],
            ["Robo total o parcial del vehículo", "Valor venal", "300 €"],
            ["Incendio del vehículo", "Valor venal", "Sin franquicia"],
            ["Lunas (parabrisas, laterales y trasera)", "Sin límite", "Sin franquicia"],
            ["Asistencia en viaje desde kilómetro 0", "Incluida", "Sin franquicia"],
        ],
        exclusiones=[
            ("Circunstancias del conductor", [
                "Siniestros conduciendo bajo <b>influencia de bebidas alcohólicas</b>, drogas o estupefacientes.",
                "Siniestros cuando el conductor <b>carezca de permiso de conducción</b> en vigor.",
                "Siniestros en <b>carreras, apuestas o pruebas deportivas</b> y sus entrenamientos.",
            ]),
            ("Uso del vehículo", [
                "El uso como <b>autoescuela, alquiler sin conductor, taxi o VTC</b>, salvo declaración expresa.",
                "El transporte de <b>mercancías peligrosas</b> o de más ocupantes de los autorizados.",
                "Los daños circulando por <b>vías no aptas</b> para la circulación o fuera de calzada.",
            ]),
            ("Daños no indemnizables", [
                "El <b>desgaste, uso o defecto de conservación</b> de las piezas.",
                "Los daños <b>exclusivamente estéticos</b> que no afecten a la seguridad.",
                "La <b>depreciación</b> del vehículo tras la reparación.",
            ]),
        ],
        franquicias=[["Conductor > 25 años y > 2 años de carné", "300 €", "Sin franquicia en lunas"],
                     ["Conductor novel (< 2 años de carné)", "600 €", "600 € en daños propios"],
                     ["Conductor ocasional no declarado", "900 €", "900 € en daños propios"]],
    ),
    dict(
        path="SALUD_FAMILIAR.pdf", producto="Salud Familiar", codigo="SF-2026-02", version="1.4",
        coberturas=[
            ["Medicina primaria y especialidades", "Sin límite", "Sin franquicia"],
            ["Pruebas diagnósticas (analítica, radiología)", "Sin límite", "Sin franquicia"],
            ["Hospitalización y cirugía en centros concertados", "Sin límite", "Sin franquicia"],
            ["Urgencias 24 h en cuadro médico", "Sin límite", "Sin franquicia"],
            ["Fisioterapia y rehabilitación", "30 sesiones/año", "10 € por sesión"],
            ["Psicología clínica", "20 sesiones/año", "15 € por sesión"],
        ],
        exclusiones=[
            ("Periodos de carencia", [
                "Las <b>intervenciones quirúrgicas</b> tienen una carencia de <b>seis (6) meses</b>.",
                "El <b>parto y la asistencia al embarazo</b> tienen una carencia de <b>diez (10) meses</b>.",
                "Los <b>tratamientos de reproducción asistida</b> tienen una carencia de <b>veinticuatro "
                "(24) meses</b> y se limitan a tres ciclos.",
            ]),
            ("Prestaciones no cubiertas", [
                "Las <b>enfermedades preexistentes</b> no declaradas en el cuestionario de salud.",
                "La <b>cirugía estética</b> y todo tratamiento sin finalidad terapéutica.",
                "Los tratamientos de <b>odontología</b> salvo extracción y limpieza anual.",
                "Los <b>medicamentos y prótesis</b> no incluidos en el catálogo.",
                "La asistencia <b>fuera del cuadro médico</b>, salvo urgencia vital acreditada.",
            ]),
        ],
        franquicias=[["Modalidad sin copago", "Sin franquicia", "Sin franquicia"],
                     ["Modalidad con copago", "Según acto médico", "10 € consulta / 25 € urgencia"],
                     ["Modalidad reembolso", "20% del gasto", "Límite 60.000 €/año"]],
    ),
]

os.makedirs("polizas", exist_ok=True)
for p in POLIZAS:
    construir_poliza(os.path.join("polizas", p["path"]), p["producto"], p["codigo"],
                     p["version"], p["coberturas"], p["exclusiones"], p["franquicias"])
print(f"✅ {len(POLIZAS)} pólizas generadas en ./polizas/")

---
# 2 · Extraer texto y trocear ⏱️ ~8 min

> 🧠 **Nota sobre la extracción.** En el [curso 3](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_pdf_ocr_opensource.ipynb) vimos cómo extraer texto de PDFs **escaneados** con un OCR open-source (Unlimited-OCR). Aquí, como nuestras pólizas tienen texto nativo y queremos **centrarnos en lo nuevo** (el vector store y el LLM propios), extraemos con **`pypdf`** — simple, local y sin gastar GPU. Si tus documentos fueran escaneos, sustituirías esta celda por el bloque de OCR del curso 3. *(Además, ese OCR fija una versión de `transformers` que no convive con la que pide Gemma 4 — otra razón para separarlos.)*

▶️ **Qué hace esta celda:** lee el texto de cada PDF, **página a página** (para poder citar la página).

In [ ]:
from pypdf import PdfReader

paginas_por_doc = {}
for p in POLIZAS:
    nombre = p["path"].replace(".pdf", "")
    reader = PdfReader(f"polizas/{p['path']}")
    paginas_por_doc[nombre] = [pg.extract_text() or "" for pg in reader.pages]
print("✅ Texto extraído de", len(paginas_por_doc), "pólizas")

▶️ **Qué hace esta celda:** el troceador. Recorre el texto línea a línea, detecta los **encabezados** (por `#` de markdown o por el patrón de sección `4. EXCLUSIONES`) y arma chunks **anteponiendo a cada uno su sección** — el truco que evita que un fragmento de exclusiones llegue al buscador sin su etiqueta. Rastrea la página.

In [ ]:
import re
import pandas as pd

def chunkear_texto(paginas, documento, max_chars=900):
    """Trocea texto plano por página respetando encabezados y anteponiendo la sección."""
    filas, heading, n, buf = [], "", 0, []
    es_heading = re.compile(
        r"^\s*(?:#{1,6}\s+(.+)|(\d+(?:\.\d+)*\.?\s+[A-ZÁÉÍÓÚÑ][^\n]{2,60}))\s*$")
    def emit(pagina):
        nonlocal n, buf
        texto = " ".join(" ".join(buf).split()).strip(); buf = []
        for i in range(0, len(texto), max_chars):
            n += 1
            trozo = texto[i:i + max_chars]
            filas.append({"documento": documento, "chunk_id": f"{documento}-{n:04d}",
                          "pagina": pagina, "heading": heading,
                          "content": f"[{heading}] {trozo}" if heading else trozo})
    for pagina, texto in enumerate(paginas, start=1):
        for linea in texto.split("\n"):
            m = es_heading.match(linea)
            if m:
                emit(pagina); heading = (m.group(1) or m.group(2)).strip()
            elif linea.strip():
                buf.append(linea)
        emit(pagina)
    return filas

filas = []
for nombre, paginas in paginas_por_doc.items():
    filas += chunkear_texto(paginas, nombre)
df_chunks = pd.DataFrame(filas)
print(f"✅ {len(df_chunks)} chunks")
df_chunks.groupby("documento").size().to_frame("chunks")

▶️ **Qué hace esta celda:** el momento de la verdad. Los chunks sobre **agua** de Hogar Plus, con su sección.

In [ ]:
pd.set_option("display.max_colwidth", 90)
agua = df_chunks[(df_chunks.documento == "HOGAR_PLUS") &
                 (df_chunks.content.str.contains("agua", case=False))]
agua.assign(extracto=agua.content.str[:110])[["pagina", "heading", "extracto"]]

> 🚩 **CHECKPOINT 2** — Deberías ver chunks de "agua" bajo secciones distintas (coberturas y exclusiones). Esa distinción es la que salvará al RAG.

---
# 3 · Levantar Postgres + pgvector en una VM ⏱️ ~20 min

Aquí montamos **nuestro almacén de vectores**. En vez de BigQuery, una **VM de Compute Engine** con **PostgreSQL** y la extensión **`pgvector`** (que añade el tipo `vector` y la búsqueda por similitud a Postgres).

**Tres piezas:**
1. Un **startup-script** que instala y configura Postgres+pgvector al arrancar la VM.
2. La **VM** (con el 5432 cerrado a todo **salvo el rango de IAP** — no la exponemos a internet).
3. Un **túnel IAP** desde el Colab: conecta el `localhost:5432` del Colab con el Postgres de la VM, de forma cifrada y sin abrir puertos públicos.

> 🧠 **Por qué IAP y no una IP pública:** la IP del Colab cambia y abrir Postgres a internet es pedir problemas. IAP (Identity-Aware Proxy) tuneliza el puerto por la identidad de Google — solo tú, autenticado, llegas a la base. Defensa en capas: IAM + firewall al rango IAP + contraseña.

▶️ **Qué hace esta celda:** escribe el **startup-script** (bash) que se ejecutará dentro de la VM al arrancar. Añade el repo oficial de PostgreSQL (para fijar la versión 16 y que sea reproducible), instala `postgresql-16-pgvector`, crea la base `appdb` y el usuario `appuser`, habilita `CREATE EXTENSION vector` y deja Postgres escuchando (solo el rango IAP podrá entrar por el firewall).

In [ ]:
%%writefile startup-postgres-pgvector.sh
#!/bin/bash
set -euxo pipefail
export DEBIAN_FRONTEND=noninteractive
PGVER=16

# La contraseña se lee del metadata server (no queda escrita en el script)
DB_PASS="$(curl -s -H 'Metadata-Flavor: Google' \
  'http://metadata.google.internal/computeMetadata/v1/instance/attributes/db-password')"

# Repo oficial PostgreSQL (PGDG) -> garantiza postgresql-16 + pgvector
apt-get update
apt-get install -y curl ca-certificates postgresql-common
yes | /usr/share/postgresql-common/pgdg/apt.postgresql.org.sh
apt-get update
apt-get install -y "postgresql-${PGVER}" "postgresql-${PGVER}-pgvector"

# Base, usuario y extensión
sudo -u postgres psql -v ON_ERROR_STOP=1 <<SQL
CREATE ROLE appuser LOGIN PASSWORD '${DB_PASS}';
CREATE DATABASE appdb OWNER appuser;
SQL
sudo -u postgres psql -v ON_ERROR_STOP=1 -d appdb -c "CREATE EXTENSION IF NOT EXISTS vector;"

# Aceptar conexiones desde el rango de IAP (IAP entra por la IP interna de la VM)
CONF="/etc/postgresql/${PGVER}/main"
echo "listen_addresses = '*'" >> "${CONF}/postgresql.conf"
echo "host all all 35.235.240.0/20 scram-sha-256" >> "${CONF}/pg_hba.conf"
systemctl restart postgresql

# Señal de "listo" para poder hacer poll desde el Colab
curl -s -X PUT --data "ready" -H 'Metadata-Flavor: Google' \
  "http://metadata.google.internal/computeMetadata/v1/instance/guest-attributes/startup/status"

▶️ **Qué hace esta celda:** crea la VM (`e2-medium`, Debian 12), le pasa la contraseña como *metadata* y el startup-script; abre el firewall **solo al rango IAP** en el 5432; y te concede el rol para tunelizar. La contraseña es **aleatoria** y vive solo en esta sesión.

In [ ]:
import secrets, subprocess

DB_PASS = secrets.token_urlsafe(20)     # contraseña aleatoria de esta sesión
ZONE    = "europe-west1-b"
VM      = "pg-vector-vm"
EMAIL   = subprocess.run(["gcloud", "config", "get-value", "account"],
                         capture_output=True, text=True).stdout.strip()

# 1) Crear la VM (con IP pública para que el apt del startup funcione; el 5432 lo cerramos abajo)
!gcloud compute instances create {VM} --zone={ZONE} --machine-type=e2-medium \
    --image-family=debian-12 --image-project=debian-cloud \
    --metadata=db-password={DB_PASS} \
    --metadata-from-file=startup-script=startup-postgres-pgvector.sh --quiet

# 2) Firewall: permitir 5432 SOLO desde el rango de IAP (no desde internet)
!gcloud compute firewall-rules create allow-iap-postgres \
    --direction=INGRESS --action=allow --rules=tcp:5432 \
    --source-ranges=35.235.240.0/20 --quiet 2>/dev/null || echo "(la regla ya existía)"

# 3) Permiso para abrir el túnel IAP
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member=user:{EMAIL} --role=roles/iap.tunnelResourceAccessor \
    --condition=None --quiet > /dev/null
print("✅ VM creada · firewall (solo IAP) · permiso de túnel concedido")

▶️ **Qué hace esta celda:** abre el **túnel IAP** en segundo plano (`localhost:5432` → Postgres de la VM) y **espera a que Postgres esté realmente listo**. Ojo: "VM creada" ≠ "Postgres listo" — el startup-script tarda **2-4 min** en instalarlo, así que reintentamos la conexión hasta que responde. ☕

In [ ]:
import socket, time, os, signal, psycopg2

LOCAL_PORT = 5432

# 1) Túnel IAP en background (grupo de proceso propio para poder cerrarlo limpio)
tunnel = subprocess.Popen(
    ["gcloud", "compute", "start-iap-tunnel", VM, "5432",
     f"--local-host-port=localhost:{LOCAL_PORT}", f"--zone={ZONE}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, preexec_fn=os.setsid)

# 2) Esperar a que el puerto local escuche (el túnel arrancó)
def port_open(host, port, timeout=120):
    t0 = time.time()
    while time.time() - t0 < timeout:
        with socket.socket() as s:
            s.settimeout(2)
            if s.connect_ex((host, port)) == 0:
                return True
        time.sleep(2)
    return False
assert port_open("127.0.0.1", LOCAL_PORT), "El túnel IAP no abrió el puerto local"

# 3) Esperar a que Postgres (startup-script) responda de verdad
def wait_db(timeout=420):
    t0, last = time.time(), None
    while time.time() - t0 < timeout:
        try:
            return psycopg2.connect(host="127.0.0.1", port=LOCAL_PORT, dbname="appdb",
                                    user="appuser", password=DB_PASS, connect_timeout=3)
        except Exception as e:
            last = e; time.sleep(6)
    raise TimeoutError(f"Postgres no respondió (¿el startup-script sigue instalando?): {last}")

print("⏳ Esperando a que la VM instale Postgres + pgvector (2-4 min)...")
conn = wait_db(); conn.autocommit = True
cur = conn.cursor()
cur.execute("SELECT version()"); print("✅ Conectado:", cur.fetchone()[0][:45], "...")
cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
print("✅ pgvector disponible")

> 🚩 **CHECKPOINT 3** — `✅ Conectado` + `✅ pgvector disponible`. Si `wait_db` agota el tiempo, el startup-script aún instala: reejecuta esta celda (el túnel sigue vivo). Si falla el túnel, reejecuta la celda anterior y esta.

---
# 4 · Embeddings open-source → pgvector ⏱️ ~12 min

Sustituimos `ML.GENERATE_EMBEDDING` de Vertex por **`BAAI/bge-m3`**, un modelo de embeddings **open-source (MIT)** que corre en la GPU del Colab. Ventajas para nuestro caso: multilingüe (bueno en español), **1024 dimensiones**, contexto largo (8192 tokens, no trunca cláusulas) y **no necesita prefijos** raros.

▶️ **Qué hace esta celda:** carga `bge-m3`, crea la tabla en Postgres con una columna **`vector(1024)`**, vectoriza los chunks y los inserta. Luego crea un **índice HNSW** (búsqueda vectorial rápida y aproximada).

In [ ]:
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer("BAAI/bge-m3", device="cuda")
print("⏳ Vectorizando", len(df_chunks), "chunks...")
embs = emb_model.encode(df_chunks["content"].tolist(),
                        normalize_embeddings=True, batch_size=16, show_progress_bar=True)

# Tabla con columna vector(1024) — la dimensión de bge-m3
cur.execute("DROP TABLE IF EXISTS polizas_chunks")
cur.execute("""
    CREATE TABLE polizas_chunks (
        chunk_id  text PRIMARY KEY,
        documento text,
        pagina    int,
        heading   text,
        content   text,
        embedding vector(1024)
    )
""")

# Insertar (el literal de un vector en pgvector es '[0.1,0.2,...]')
for (_, row), emb in zip(df_chunks.iterrows(), embs):
    lit = "[" + ",".join(f"{x:.6f}" for x in emb) + "]"
    cur.execute("INSERT INTO polizas_chunks VALUES (%s,%s,%s,%s,%s,%s)",
                (row.chunk_id, row.documento, int(row.pagina), row.heading, row.content, lit))

# Índice HNSW para búsqueda por coseno
cur.execute("CREATE INDEX ON polizas_chunks USING hnsw (embedding vector_cosine_ops)")
cur.execute("SELECT COUNT(*) FROM polizas_chunks")
print(f"✅ {cur.fetchone()[0]} chunks vectorizados y guardados en Postgres/pgvector")

> 🚩 **CHECKPOINT 4** — El número de chunks coincide con el del bloque 2. Ya tienes tu vector store **propio**, con los embeddings dentro.

---
# 5 · Búsqueda vectorial en SQL ⏱️ ~8 min

El `VECTOR_SEARCH` de BigQuery se convierte en **SQL de Postgres**. El operador **`<=>`** es la distancia coseno: `ORDER BY embedding <=> consulta LIMIT k` te da los k chunks más cercanos.

▶️ **Qué hace esta celda:** la función de búsqueda. Vectoriza la pregunta con el mismo `bge-m3` y pide a Postgres los k chunks más próximos, con filtro opcional por póliza (búsqueda híbrida).

In [ ]:
def buscar_clausulas(pregunta, k=5, documento=None):
    q = emb_model.encode([pregunta], normalize_embeddings=True)[0]
    q_lit = "[" + ",".join(f"{x:.6f}" for x in q) + "]"
    where = "WHERE documento = %(doc)s" if documento else ""
    sql = f"""
        SELECT documento, pagina, heading, content,
               ROUND((embedding <=> %(q)s)::numeric, 4) AS distancia
        FROM polizas_chunks
        {where}
        ORDER BY embedding <=> %(q)s
        LIMIT %(k)s
    """
    params = {"q": q_lit, "k": k}
    if documento:
        params["doc"] = documento
    cur.execute(sql, params)
    return pd.DataFrame(cur.fetchall(),
                        columns=["documento", "pagina", "heading", "content", "distancia"])

# 🎯 la pregunta del millón: mira los HEADINGS que recupera
buscar_clausulas("¿me cubre el agua de lluvia que ha entrado en casa?",
                 k=4, documento="HOGAR_PLUS")[["documento", "pagina", "heading", "distancia"]]

> 🎓 Como en los cursos anteriores, el buscador recupera chunks de **coberturas y de exclusiones** a la vez (semánticamente los dos hablan de agua). La distinción la pondrá el **heading**, que viaja al LLM. Y todo esto ocurre ya **en tu Postgres**, no en BigQuery.

---
# 6 · RAG con Gemma 4 en tu GPU ⏱️ ~15 min

Y el último tercero fuera: en vez de Gemini, **Gemma 4 (E4B)** —el modelo abierto de Google (licencia Apache 2.0)— corriendo en la GPU del Colab. Lo cargamos en **4-bit** para que quepa en una T4 junto al embedder.

▶️ **Qué hace esta celda:** carga Gemma 4 E4B cuantizado a 4-bit. La primera vez descarga los pesos (unos minutos). En T4 usamos `float16` como tipo de cómputo y atención `eager` (la T4 no tiene FlashAttention).

> 💡 Si prefieres el clásico **`google/gemma-3-4b-it`** (4B denso), cambia `GEMMA_ID`. Ojo: ese sí está *gated* — tendrías que aceptar su licencia en Hugging Face y hacer `login()` con un token.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

GEMMA_ID = "google/gemma-4-E4B-it"     # Apache 2.0, sin token. Alternativa: "google/gemma-3-4b-it"

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

print("⏳ Cargando Gemma en 4-bit (descarga los pesos la primera vez)...")
gemma_tok = AutoTokenizer.from_pretrained(GEMMA_ID)
gemma = AutoModelForCausalLM.from_pretrained(
    GEMMA_ID, quantization_config=bnb, device_map="auto", attn_implementation="eager")
print("✅ Gemma cargado")

▶️ **Qué hace esta celda:** el RAG completo, ahora **100% local**. Recupera de tu Postgres, monta el contexto con la procedencia (póliza, página, sección) y **Gemma** genera la respuesta citando. Usamos `apply_chat_template` (la forma correcta de hablar con Gemma) y generación *greedy* (respuestas estables).

In [ ]:
INSTR = ("Eres el asistente de atención al cliente de Peñalara Seguros. "
         "Responde SOLO con la información del CONTEXTO. "
         "Antes de afirmar que algo está cubierto, comprueba la SECCIÓN del fragmento: "
         "si viene de EXCLUSIONES, NO está cubierto, aunque hable del mismo riesgo. "
         "Cita siempre la póliza, la página y la cláusula. "
         "Si el contexto no basta, di que no consta en la póliza. No inventes.")

def responder(pregunta, k=6, documento=None, verbose=True):
    docs = buscar_clausulas(pregunta, k, documento)
    contexto = "\n\n---\n\n".join(
        f"[Póliza: {r.documento} | Página: {r.pagina} | Sección: {r.heading}]\n{r.content}"
        for r in docs.itertuples())
    messages = [{"role": "user",
                 "content": f"{INSTR}\n\n### CONTEXTO:\n{contexto}\n\n### PREGUNTA:\n{pregunta}"}]
    inputs = gemma_tok.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True).to(gemma.device)
    out = gemma.generate(**inputs, max_new_tokens=512, do_sample=False)
    resp = gemma_tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    if verbose:
        print(f"🔎 {len(docs)} fragmentos | secciones: {[s for s in docs.heading.unique() if s]}\n")
    return resp.strip()

# 🎯 la pregunta trampa
print(responder("Ha entrado agua de lluvia por la terraza y se me ha estropeado el parqué. "
                "¿Me lo cubre el seguro de hogar?", documento="HOGAR_PLUS"))

> 🎓 Si todo fue bien, Gemma responde **NO cubierto**, citando la cláusula de exclusiones — igual que hacía Gemini, pero ahora **sin que nada saliera de tu infraestructura**: el texto lo extrajo `pypdf`, lo vectorizó `bge-m3`, lo buscó tu Postgres y lo respondió tu Gemma.

▶️ **Qué hacen estas dos celdas:** más preguntas, sobre otras pólizas.

In [ ]:
print(responder("Contraté el seguro de salud hace 3 meses y necesito operarme del menisco. "
                "¿Me lo cubren ya?", documento="SALUD_FAMILIAR"))

In [ ]:
print(responder("Tuve un accidente y el coche lo conducía mi sobrino, que sacó el carné hace 8 meses "
                "y no está declarado en la póliza. ¿Qué franquicia me toca pagar?",
                documento="AUTO_TODO_RIESGO"))

> 🎓 **Anti-alucinación:** prueba `responder("¿Cubre la póliza un ataque de dragones?")`. Con las reglas del prompt, Gemma debería decir que no consta en vez de inventarse una cláusula.

---
# 7 · Comparativa: self-hosted vs. gestionado ⏱️ ~6 min

Ya tienes las dos arquitecturas montadas (esta y la de los cursos 1-2). Toca la decisión honesta.

| Eje | Gestionado (BigQuery + Vertex + Gemini) | Self-hosted (este curso) |
|---|---|---|
| **Privacidad del dato** | sale a las APIs de Google | **no sale de tu perímetro** |
| **Coste** | por uso (consultas, tokens, páginas) | por **infraestructura** (VM/GPU encendidas), fijo |
| **Escalado** | automático, infinito | lo gestionas tú (réplicas, sharding) |
| **Puesta en marcha** | minutos, cero infra | horas: VMs, drivers, versiones, red |
| **Mantenimiento** | de Google | **tuyo** (parches, backups, monitorización) |
| **Calidad del modelo** | Gemini (frontera) | Gemma 4 (muy bueno, pero un escalón por debajo) |
| **Dependencia** | del proveedor y su roadmap | de nadie: guardas los pesos |
| **Latencia** | red + API | local (rápido si tienes GPU) |

### 🎓 Cuándo cada uno

- **Gestionado** si: prototipas rápido, tu dato no es sensible, el volumen es irregular, y no quieres equipo de infra.
- **Self-hosted** si: **el dato no puede salir** (seguros, salud, banca, sector público), tienes volumen alto y sostenido (el coste fijo compensa), o necesitas **soberanía** (evitar lock-in, poder auditar y versionar los pesos).

> No hay respuesta única. Lo valioso es **saber montar las dos** y elegir por la restricción que más apriete. En seguros, muy a menudo, esa restricción es la del dato — y por eso este curso existe.


---
# 8 · Evaluación y limpieza ⏱️ ~8 min

### Mini-evaluación: ¿acierta el sentido?

▶️ **Qué hace esta celda:** casos con respuesta conocida, donde solo la jerarquía desempata cobertura de exclusión — ahora respondidos por tu stack self-hosted.

In [ ]:
casos = [
    ("Se ha roto una tubería del baño y ha inundado el salón. ¿Está cubierto?",
     "HOGAR_PLUS", "SÍ (cláusula 3.2, rotura accidental de conducciones)"),
    ("Entra agua de lluvia por una filtración en la terraza. ¿Está cubierto?",
     "HOGAR_PLUS", "NO (cláusula 4.1.a, filtraciones aunque sean por lluvia)"),
    ("Dejé la ventana abierta, llovió y se estropeó el suelo. ¿Está cubierto?",
     "HOGAR_PLUS", "NO (cláusula 4.1.d, agua por huecos dejados abiertos)"),
    ("Necesito una operación de rodilla a los 8 meses de contratar. ¿Cubierta?",
     "SALUD_FAMILIAR", "SÍ (carencia quirúrgica de 6 meses ya superada)"),
    ("Quiero una rinoplastia estética. ¿La cubre el seguro de salud?",
     "SALUD_FAMILIAR", "NO (exclusión: cirugía estética sin fin terapéutico)"),
]

for pregunta, doc, esperado in casos:
    print("═" * 78)
    print(f"❓ {pregunta}")
    print(f"🎯 Esperado: {esperado}")
    print(f"🤖 {responder(pregunta, documento=doc, verbose=False)[:300]}...\n")

### 🧪 Autoevaluación

1. ¿Qué reemplaza a BigQuery en este curso, y con qué tipo de columna guardamos los vectores?
2. ¿Por qué conectamos a Postgres por un **túnel IAP** en vez de abrir el 5432 a internet?
3. `bge-m3` produce vectores de 1024 dimensiones. ¿Dónde se refleja ese número en el código?
4. ¿Qué operador de pgvector usamos para la búsqueda y qué mide?
5. Enumera, de tu pipeline, qué pieza corre **en tu GPU** y qué pieza corre **en la VM**.

### Ejercicios para casa

- **Fácil** — Cambia `bge-m3` por `Qwen/Qwen3-Embedding-0.6B` (recuerda: sus *queries* usan `prompt_name="query"`). ¿Mejora el retrieval?
- **Medio** — Añade un **índice IVFFlat** en vez de HNSW y compara. Con pocos datos, ¿se usa el índice o hace *seq scan*?
- **Medio** — Mete 5.000 chunks sintéticos y mide el tiempo de búsqueda **con y sin** el índice HNSW (`SET enable_seqscan = off/on`).
- **Difícil** — Sirve Gemma como **API** (Ollama o vLLM) en una segunda VM con GPU y haz que el RAG la consuma por HTTP, separando la GPU del orquestador.
- **Difícil** — Sustituye Gemma por tu **adapter fine-tuneado** del cuaderno siguiente y compara las respuestas.

### 🧹 Limpieza — ⚠️ IMPORTANTE

> Una VM encendida **cuesta dinero mientras exista** (a diferencia de BigQuery, que cobra por consulta). **Ejecuta esta celda al terminar** para borrarla. Pon `BORRAR = True`.

In [ ]:
BORRAR = False   # ⚠️ cámbialo a True y ejecuta para borrar la VM (si no, seguirá costando)

import gc
# Cerrar el túnel IAP
try:
    os.killpg(os.getpgid(tunnel.pid), signal.SIGTERM); print("🔌 Túnel IAP cerrado")
except Exception:
    pass

if BORRAR:
    !gcloud compute instances delete {VM} --zone={ZONE} --quiet
    !gcloud compute firewall-rules delete allow-iap-postgres --quiet
    print("🗑️  VM y regla de firewall borradas")
else:
    print("⚠️  BORRAR = False. La VM SIGUE ENCENDIDA y generando coste.")
    print(f"    Para apagarla: gcloud compute instances delete {VM} --zone={ZONE}")

# Liberar la GPU
try:
    del gemma, emb_model; gc.collect(); torch.cuda.empty_cache(); print("🧹 GPU liberada")
except Exception:
    pass

---
## 🧭 Mapa final: qué te llevas

1. **La soberanía se monta pieza a pieza.** Vector store (`pgvector`), embeddings (`bge-m3`) y generación (Gemma) son sustituibles uno a uno; no hace falta cambiarlo todo de golpe.
2. **`pgvector` convierte Postgres en un vector store** perfectamente capaz: el `VECTOR_SEARCH` es un `ORDER BY embedding <=> consulta`.
3. **Conectar a infra propia con seguridad es parte del trabajo**: IAP + firewall al rango correcto + contraseña, en vez de abrir puertos.
4. **El pipeline de RAG no cambió**: recuperar → aumentar → generar con citas. Solo cambió **dónde vive cada pieza** — que es, al final, la única pregunta de este curso.

### 📚 Para seguir
- [pgvector](https://github.com/pgvector/pgvector) · [bge-m3](https://huggingface.co/BAAI/bge-m3) · [Gemma 4](https://huggingface.co/google/gemma-4-E4B-it)
- **Siguiente:** fine-tuning de Gemma 4 para que responda con el estilo de Peñalara y se enchufe a este mismo RAG.
